# Project: Investigate a Dataset - TMDb Movie Data

## Table of Contents
<ul>
<li><a href="#intro">Introduction</a></li>
<li><a href="#wrangling">Data Wrangling</a></li>
<li><a href="#eda">Exploratory Data Analysis</a></li>
<li><a href="#conclusions">Conclusions</a></li>
</ul>

<a id='intro'></a>
## Introduction

### Dataset Description 

This dataset contains information about 10,000 movies collected from The Movie Database (TMDb), including user ratings and revenue. 

The dataset contains the following columns:  

| Column | Description |
|:-------|:------------|
| **id** | Unique identifier for each record |
| **imdb_id** | Unique identifier from the IMDb website |
| **popularity** | Score from TMDb indicating popularity based on page views, votes, and other engagement signals |
| **budget** | Production budget in U.S. dollars |
| **revenue** | Worldwide box-office revenue |
| **original_title** | Original title of the movie |
| **cast** | List of main cast members |
| **homepage** | Official movie homepage |
| **director** | Movie director name(s) |
| **tagline** | Marketing tagline |
| **keywords** | List of keywords associated with the movie |
| **overview** | Short plot summary or synopsis |
| **runtime** | Movie duration in minutes |
| **genres** | List of genres associated with the movie |
| **production_companies** | List of production companies associated with the movie |
| **release_date** | Date the movie was released |
| **vote_count** | Number of user votes the movie received on TMDb |
| **vote_average** | Average user rating on TMDb (0-10 scale) |
| **release_year** | Year the movie was released |
| **budget_adj** | Budget adjusted for inflation (expressed in 2010 dollars) |
| **revenue_adj** | Revenue adjusted for inflation (expressed in 2010 dollars) |

### Question(s) for Analysis
&nbsp;&nbsp;&nbsp;&nbsp;**1. What factors are associated with higher profit?**

&nbsp;&nbsp;&nbsp;&nbsp;**2. How has movie quality changed over time?**

&nbsp;&nbsp;&nbsp;&nbsp;**3. Is there a relationship between budget and rating?**

In [ ]:
import pandas as pd
import numpy as np

<a id='wrangling'></a>
## Data Wrangling


In [ ]:
df = pd.read_csv('Database_TMDb_movie_data/tmdb-movies.csv')
df.head()

First, I will use the info method to get a basic understanding of the dataset. Here, I can see the column names, data types, and non-null counts.

In [ ]:
df.info()

Next, I will remove irrelevant columns and verify their removal.

In [ ]:
df.drop(['id', 'imdb_id', 'popularity', 'budget', 'revenue', 'homepage', 'tagline', 'keywords', 'overview'], inplace=True, axis=1)

In [ ]:
df.info()

Next, I'll use the describe method to identify any potential problems in the data.

In [ ]:
df.describe()

The release_date column also needs to be converted to datetime format. This will be done with pandas' built-in to_datetime method. There also appear to be several missing values in cast, director, genres, runtime, and production_companies; these rows can be removed.

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'])

In [ ]:
df.describe(datetime_is_numeric=True)

This presents several problems. The release_date column shows movies released in 2075, and some movies have $0 for budget and revenue. These will need to be addressed in the cleaning phase. I will need to look at release dates after 2015-01-01 since 2015 is the max year.

In [ ]:
df[df['release_date'] > '2015-01-01'][['release_date', 'release_year']]

Now the max year is showing 2066. The discrepancy was likely caused by how pandas interprets two-digit years in release_date. A date like 12/25/75 becomes 1975, but a date like 12/25/15 becomes 2015, and something like 12/25/75 can become 2075. However, the problem is now clear. I just need to make sure the year in release_year matches the year in the year column. I will do this by creating a mask where the parsed year doesn't match release_year. Then I will fix those dates by replacing the year with the correct release_year.

In [ ]:
df['original_title'].duplicated().sum()

There are 295 duplicate movies in the dataset. These can be dropped.

In [ ]:
df[df['budget_adj'] == 0]['budget_adj'].count()

In [ ]:
df[df['revenue_adj'] == 0]['revenue_adj'].count()

In [ ]:
df[df['runtime'] == 0]['runtime'].count()

5696 records are missing budget information, and 6016 records are missing revenue information. This indicates missing data. Since the analysis needs to address profitability, these will need to be dropped. This would remove a very significant portion of the data, so I will create a second dataframe for the profit analysis and add a profit column to it. Budget and revenue can be dropped from the original dataframe. This will be done during the cleaning phase so that the cleaning only needs to be done on one dataframe.

In [ ]:
df.describe()


### Data Cleaning
 

First, I will clean the release_date column by creating a mask that flags rows where the parsed year doesn't match the release year. Then I will fix those dates by replacing them with the correct release year from the release_year column.

In [ ]:
mask = df['release_date'].dt.year != df['release_year']
df.loc[mask, 'release_date'] = df.loc[mask].apply(
    lambda row: row['release_date'].replace(year=int(row['release_year'])), axis=1)

clean_df = df[df['release_date'] > '2015-01-01'][['release_date', 'release_year']]

Now that the dates are cleaned, I will drop the duplicates.

In [ ]:
clean_df = df.drop_duplicates(subset='original_title')

Now that the duplicates have been dropped, I will drop the null values and make a copy without the missing budget and revenue fields. These columns will then be dropped from the original dataframe. I will then add the profit column to the new dataframe.

In [ ]:
clean_df = clean_df.dropna()

In [ ]:
clean_df = clean_df[clean_df['runtime'] > 0]

In [ ]:
profit_df = clean_df[(clean_df['budget_adj'] > 0) & (clean_df['revenue_adj'] > 0)].copy()

In [ ]:
profit_df['profit'] = profit_df['revenue_adj'] - profit_df['budget_adj']

In [ ]:
clean_df = clean_df.drop(['budget_adj', 'revenue_adj'], axis=1)

Now, I can reset the index for both dataframes to give them a cleaner look and to avoid problems with copies.

In [ ]:
clean_df.reset_index(inplace=True)
profit_df.reset_index(inplace=True);

I will call info and describe one last time and check for duplicates on both dataframes to make sure everything looks good.

In [ ]:
clean_df.info()

In [ ]:
profit_df.info()

In [ ]:
clean_df.describe(datetime_is_numeric=True)

In [ ]:
profit_df.describe(datetime_is_numeric=True)

In [ ]:
clean_df['original_title'].duplicated().sum()

In [ ]:
profit_df['original_title'].duplicated().sum()

Out of 10866 rows, clean_df now has 9477 rows and profit_df has 3708 rows. There are no duplicates and no missing values in either dataframe.

<a id='eda'></a>
## Exploratory Data Analysis


### Question 1: What factors are associated with higher profit?

The first thing I want to look at is what factors are associated with higher profits. I’m going to look at the distribution of profit, profit versus runtime, profit averages by genre, director, cast members, and production company. I will use a function to split the pipe-delimited columns for genre, director, cast, and production company.

In [ ]:
def split_pipe_column(series):
    return series.str.split('|').explode().str.strip()

In [ ]:
profit_df['profit'].plot.hist(bins=20, 
                              figsize=(10, 5), 
                              title='Distribution of Profit\n(billions)');

In [ ]:
profit_stats = profit_df['profit'].describe()
print(f'Profit STD:    ${profit_stats[2].round(2):,}')
print(f'Profit Mean:   ${profit_stats[1].round(2):,}')
print(f'Profit Min:    ${profit_stats[3].round(2):,}')
print(f'Profit Q1:     ${profit_stats[4].round(2):,}')
print(f'Profit Median: ${profit_stats[5].round(2):,}')
print(f'Profit Q3:     ${profit_stats[6].round(2):,}')
print(f'Profit Max:    ${profit_stats[7].round(2):,}')

The distribution of profit is heavily right-skewed. Most movies earn relatively modest profits or losses, while a small number of very high-grossing movies pull the mean well above the median and create a large standard deviation.

Next, I want to look at profit versus runtime. Maybe longer or shorter movies generate more profit? 

In [ ]:
profit_df.plot.scatter(x='runtime', 
                       y='profit', 
                       figsize=(10,5),
                       title='Profit v. Runtime',
                       xlabel='Runtime\n(minutes)',
                       ylabel='Profit\n(billions)');

In [ ]:
runtime_stats = profit_df['runtime'].describe()
print(f'Runtime Q1:     {runtime_stats[4].round(2)}')
print(f'Runtime Median: {runtime_stats[5].round(2)}')
print(f'Runtime Q3:     {runtime_stats[6].round(2)}')

Longer or shorter movies are not associated with higher profit. There is no strong relationship between runtime and profit. Movies across a wide range of runtimes can be highly profitable or unprofitable. While most films fall between about 96 and 119 minutes, this is simply the typical runtime range in the data, not evidence of a profitability sweet spot. 

Next, I want to look at the average profit by genre. Maybe some genres do better than others?

In [ ]:
genre_df = profit_df[['profit']].join(split_pipe_column(profit_df['genres']).rename('genre'))
top5_genre = (
    genre_df.groupby('genre')['profit']
    .mean()
    .reset_index()
    .sort_values('profit', ascending=False)
    .head(5)
)
top5_genre.plot.bar(x='genre', 
                    y='profit', 
                    rot=True, 
                    figsize=(10,5),
                    title='Avg Profit by Genre',
                    xlabel='Genre',
                    ylabel='Avg Profit\n(hundreds of millions)',
                    legend=False);

Animation, Adventure, Fantasy, Family, and Science Fiction have the highest average profits, ranging from roughly \$140 million to \$200 million. These genres outperform the others in this dataset on average.

Next, I want to look at the average profit by director. Maybe some directors produce better movies than others?

In [ ]:
director_df = profit_df[['profit']].join(split_pipe_column(profit_df['director']))
top5_director = (
    director_df.groupby('director')['profit']
    .mean()
    .reset_index()
    .sort_values('profit', ascending=False)
    .head(5)
)
top5_director.plot.bar(x='director', 
                       y='profit', 
                       rot=True, 
                       figsize=(10,5),
                       title='Avg Profit by Director',
                       xlabel='Director',
                       ylabel='Avg Profit\n(billions)',
                       legend=False);

Irwin Winkler, Clyde Geronimi, Hamilton Luske, Richard Marquand, and George Lucas have the highest average profits, ranging from roughly \$1 billion to \$1.5 billion. These directors outperform the others in this dataset on average. It’s important to note that the average profit by director can be influenced by the number of movies each director has in the dataset. Directors with only one or two high-grossing movies may rank higher than more prolific directors.

Next, I want to look at the average profit by cast member. Do certain actors tend to appear in more profitable movies? 

In [ ]:
cast_df = profit_df[['profit']].join(split_pipe_column(profit_df['cast']))
top5_cast = (
    cast_df.groupby('cast')['profit']
    .mean()
    .reset_index()
    .sort_values('profit', ascending=False)
    .head(5)
)
top5_cast.plot.bar(x='cast', 
                   y='profit', 
                   rot=True, 
                   figsize=(10,5),
                   title='Avg Profit by Cast Member',
                   xlabel='Cast Member',
                   ylabel='Avg Profit\n(billions)',
                   legend=False);

Jason Miller, Robert MacNaughton, Daisy Ridley, Ken Howard, and Wendy Gazelle have appeared in movies that average higher profitability than those of other actors in the dataset, ranging from about \$1.5 billion to over \$2 billion. Again, it’s important to note that these are averages. Actors with only one or two high-grossing movies may rank higher than more prolific actors.

Finally, I want to look at the average profit by production company. Do certain production companies tend to release more profitable movies than others in the dataset? 

In [ ]:
company_df = profit_df[['profit']].join(split_pipe_column(profit_df['production_companies']).rename('company'))
top5_company = (
    company_df.groupby('company')['profit']
    .mean()
    .reset_index()
    .sort_values('profit', ascending=False)
    .head(5)
)
top5_company.plot.bar(x='company', 
                      y='profit', 
                      rot=True, 
                      figsize=(10,5),
                      title='Avg Profit by Production Company',
                      xlabel='Production Company',
                      ylabel='Profit\n(billions)',
                      legend=False);

Hoya Productions, Truenorth Productions, Alfran Productions, Argyle Enterprises, and EMI Films have the highest average profits, ranging from about \$1 billion to over \$2 billion. As with directors and cast, these averages can be influenced by companies that have only one or two very high-grossing films in the dataset.

### Question 2: How has movie quality changed over time?

The next question I would like to address is whether movie quality has changed over time. To explore this, I will look at average rating, budget, and profit by year.

In [ ]:
rating_by_year = (
    clean_df.groupby('release_year')['vote_average']
    .mean()
    .reset_index()
    .sort_values('release_year')
)
rating_by_year.plot.line(x='release_year', 
                         y='vote_average', 
                         figsize=(10,5),
                         title='Avg Rating by Year',
                         xlabel='Year',
                         ylabel='Avg Rating',
                         legend=False);

In [ ]:
max_rating = rating_by_year.iloc[rating_by_year['vote_average'].idxmax()][0]
min_rating = rating_by_year.iloc[rating_by_year['vote_average'].idxmin()][0]
print(f'Year with highest average rating: {max_rating:.0f}')
print(f'Year with lowest average rating:  {min_rating:.0f}')

After plotting average rating against release year, a clear downward trend is evident. The year with the highest average rating is 1973, and the year with the lowest average rating is 2012. This decline could reflect changes in rating behavior, changes in the types of movies included in the dataset, or other factors, rather than a pure drop in film quality. 

Next, I want to examine the average movie budget over time. Has the average budget declined along with the average ratings? 

In [ ]:
budget_by_year = (
    profit_df.groupby('release_year')['budget_adj']
    .mean()
    .reset_index()
    .sort_values('release_year')
)
budget_by_year.plot.line(x='release_year', 
                         y='budget_adj', 
                         figsize=(10,5),
                         title='Avg Budget by Year',
                         xlabel='Year',
                         ylabel='Avg Budget\n(tens of millions)',
                         legend=False);

In [ ]:
max_budget = budget_by_year.iloc[budget_by_year['budget_adj'].idxmax()]
min_budget = budget_by_year.iloc[budget_by_year['budget_adj'].idxmin()]
print(f'Year with highest average budget: {max_budget[0]:.0f} - ${max_budget[1]:,.2f}')
print(f'Year with lowest average budget:  {min_budget[0]:.0f} - ${min_budget[1]:,.2f}')

Aside from a few outlier years in the 1960s, the average movie budget has increased over time. 1972 had the lowest average budget at about \$13.3 million, and 1965 had the highest at about \$77.5 million.

Next, I’m going to look at average movie profit over time. Given the decline in average rating and the general increase in average budget, what has happened to profitability? 

In [ ]:
profit_by_year = (
    profit_df.groupby('release_year')['profit']
    .mean()
    .reset_index()
    .sort_values('release_year')
)
profit_by_year.plot.line(x='release_year', 
                         y='profit', 
                         figsize=(10,5),
                         title='Avg Profit by Year',
                         xlabel='Year',
                         ylabel='Avg Profit\n(hundreds of millions)',
                         legend=False);

In [ ]:
max_profit = profit_by_year.iloc[profit_by_year['profit'].idxmax()]
min_profit = profit_by_year.iloc[profit_by_year['profit'].idxmin()]
print(f'Year with highest average profit: {max_profit[0]:.0f} - ${max_profit[1]:,.2f}')
print(f'Year with lowest average profit:  {min_profit[0]:.0f} - ${min_profit[1]:,.2f}')

Like the average ratings, average profit has also declined over time. 1965 had the highest average profit, at about \$521.7 million, and 2006 had the lowest, at about \$56.7 million.

### Question 3: Is there a relationship between budget and rating?

Finally, the last question I would like to answer is whether there is a relationship between budget and rating. Does spending more money on a movie produce a better movie?

In [ ]:
profit_df.plot.scatter(x='budget_adj', 
                       y='vote_average', 
                       figsize=(10,5),
                       title='Budget v. Avg Rating',
                       xlabel='Budget\n(hundreds of millions)',
                       ylabel='Avg Rating');

In [ ]:
corr = np.corrcoef(profit_df['budget_adj'], profit_df['vote_average'])[0, 1]
print(f"Correlation between budget and rating: {corr:.3f}")

After plotting average rating against budget, there is no clear relationship. The correlation coefficient is 0.040, indicating a very weak positive association that is not meaningful. Spending more money on a movie is not associated with higher ratings in this dataset.

<a id='conclusions'></a>
## Conclusions

### Summary of Findings

**Question 1: What factors are associated with higher profit?**  
To answer this question, I examined profit versus runtime, profit averages by genre, director, cast members, and production company. The analysis did not find an association between runtime and profitability. Highly profitable movies did show an association with particular genres, directors, cast members, and production companies. It is important to note that these averages can be influenced by the number of movies in the dataset.

**Question 2: How has movie quality changed over time?**  
Next, I explored average rating, budget, and profit over time. The average movie rating has declined steadily since the 1970s. Conversely, average budget has risen. Despite increased spending on movie production, average profit has also declined since the 1970s.

**Question 3: Is there a relationship between budget and rating?**  
Finally, I wanted to explore a possible relationship between budget and average rating. No clear relationship was noted. The correlation coefficient was 0.040. This indicates a very weak positive association, but it is not meaningful. Spending more money on a movie is not associated with higher ratings in this dataset.

### Limitations
This analysis had several limitations. First, many movies were missing financial data, resulting in only 3708 movies being included in the financial analyses. Second, averages by director, cast, and production company can be skewed by movies with few data points. Finally, there may have been changes in how TMDb ratings or popularity are calculated over time. 

### Further Research
Additional data could help make this analysis more meaningful. It would be interesting to look at marketing data, audience demographics, critic-versus-user scores, or more comprehensive financial data.